# Dunnhumby M3 — 최종 test 실행 (교수님 지시 반영)

**검증 분할을 쓰지 않습니다.** 학습·검증 구간을 **DAY 697까지 합쳐** 학습하고, **DAY 698~704 test 구간을 마지막에 한 번만** 채점합니다. DAY 705~711은 끝까지 열지 않습니다. epoch은 300으로 고정이며 중간에 test를 보지 않습니다 — 고를 것이 없습니다.

**왜 기존 체크포인트를 못 쓰는가.** 지금까지의 개발 백테스트는 데이터를 DAY 690에서 잘라냅니다. 그 설정에서 holdout을 켜면 돌아오는 것이 **DAY 684~690 — 우리가 수십 번 본 그 구간**입니다. 백테스트 안에는 안 본 데이터가 없어서, 확증하려면 전체 기간으로 나와 다시 학습해야 합니다.

**무엇을 비교하나.** M1(이진 그래프)과 **CLV 두 축(N·V)을 함께 쓴 중심화 가치그래프**입니다. 개발분할에서 사전등록 조건을 충족한 그 설정이고, 강도 `beta`는 이 학습 구간에서 다시 역산합니다(로컬 확인값 0.2155, 개발 0.2161). 학습 전 게이트가 가격·인기·연결수 대리변수화를 막습니다(로컬 확인: 가격 −0.004, 인기 −0.016, degree +0.015).

**보고 방식.** 시드별 수치와 **시드 평균**, 그리고 각 지표를 몇 개 시드가 개선했는지를 함께 냅니다. 사전등록 조건은 **평균 기준** 두 경제지표 개선 + 여섯 정확도가 M1의 99% 이상입니다.

**비용.** 시드당 M1+M3 2회 × 약 1.7시간 ≈ **3.4시간**. 먼저 1시드를 돌리고, 그 시드는 이후 10시드에 **그대로 포함**되어 버려지지 않습니다.

결과가 불리해도 그대로 보고합니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive

# 1시드 먼저. 나중에 (42,43,44,45,46,47,48,49,50,51) 로 바꿔 다시 실행하면
# 끝난 시드는 건너뛰고 나머지만 학습한다.
SEEDS = (42,)

ROOT = Path('/content/drive/MyDrive/논문/data')
if not os.path.ismount('/content/drive'):
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 학습은 시작되지 않았습니다.') from exc

SOURCE_COMMIT = '4ff3371cbdec82827e60e0247bb7e3f4485db8b3'
REPO = Path('/content/clv-m3-dh-test1-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), \
        '다른 소스가 로드되어 있습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import lightgcn_clv_m3_dunnhumby_test1 as t1
cfg = t1.configure_dunnhumby_test1(seeds=SEEDS)
print(json.dumps(t1.preflight_summary(cfg), ensure_ascii=False, indent=2))

## 1. 분할과 게이트 확인 (학습 없음)

**분할 경계가 `train+validation ≤ 697 < test ≤ 704 < holdout`** 으로 찍혀야 합니다. 그래야 백테스트를 벗어나 안 본 구간을 평가하는 것입니다.

게이트가 한계를 넘으면 여기서 예외가 나고 학습은 시작되지 않습니다. 로컬 확인값: `beta 0.2155, CV 0.200, 가격 −0.004, 인기 −0.016, degree +0.015`.

In [ ]:
prepared = t1._prepare(cfg, SEEDS[0])
print('test 평가유저:', len(prepared['cache'].users))
arm_graph = t1.build_graph(prepared, cfg)      # 게이트 실패 시 예외
print('beta', round(arm_graph['beta'], 4))
print(json.dumps(arm_graph['audit'], ensure_ascii=False, indent=1))

## 2. 학습과 1회 평가 (시드당 약 3.4시간)

M1과 M3를 각각 300 epoch 고정으로 학습하고, **마지막에 한 번** test를 채점합니다. epoch마다 Drive에 저장하므로 끊기면 같은 셀을 다시 실행하면 이어집니다. 끝난 arm은 건너뜁니다.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU 런타임을 사용하세요.'
table = t1.run_dunnhumby_test1(cfg)

## 3. 결과와 원본 ZIP

`condition_met`이 사전등록 조건(평균 기준 두 경제지표 개선 + 여섯 정확도 99% 보호)입니다. 시드가 하나면 `paired_diff_sd`는 NaN이고, 10시드로 늘리면 채워집니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

reading = table.attrs['reading']
print(json.dumps(reading, ensure_ascii=False, indent=2))
print()
show = list(t1.ACCURACY_METRICS) + list(t1.ECONOMIC_METRICS)
print(table.set_index(['seed', 'model_id'])[show].to_string())

paths = table.attrs['result_paths']
zip_path = Path('/content/m3_dunnhumby_test1_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))